# micro-sam GPU segmentation in napari

In [1]:
import os
import sys
_qt_plugins = os.path.join(sys.prefix, "Library", "lib", "qt6", "plugins")
if os.path.isdir(_qt_plugins):
    os.environ["QT_PLUGIN_PATH"] = _qt_plugins
    os.environ["QT_QPA_PLATFORM_PLUGIN_PATH"] = os.path.join(_qt_plugins, "platforms")
else:
    print("WARNING: Qt6 plugin dir not found at", _qt_plugins)
import warnings
warnings.filterwarnings("ignore")
import torch
import napari
import micro_sam
from pathlib import Path
from micro_sam.training import train_sam
from micro_sam.util import export_custom_sam_model
import matplotlib.pyplot as plt
from skimage.color import label2rgb
import numpy as np
import imageio.v3 as imageio
from magicgui.widgets import Container, PushButton, Label
import random
from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_instance_segmentation
import imageio.v3 as imageio
from torch.utils.data import random_split
import torch_em
from torch_em.data import MinInstanceSampler
from micro_sam.training import train_sam_for_configuration, default_sam_dataset
from micro_sam.training.util import get_raw_transform
from micro_sam.sam_annotator import image_folder_annotator

print("micro_sam:", micro_sam.__version__)
print("napari   :", napari.__version__)
print("torch    :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device :", torch.cuda.get_device_name(0) if device == "cuda" else "cpu")


micro_sam: 1.8.7
napari   : 0.8.0
torch    : 2.12.1
CUDA available: True
Using device : NVIDIA GeForce RTX 5080


# Image Labelling
- Save images in a single folder on your computer
- Run the cell below, navigate to plugins -> segment anything for microscopy -> image series annotator
- Use the light microscopy model as base

In [ ]:
# GUI Only Version
viewer = napari.Viewer()

In [ ]:
# Launch from Code Version
root = r"Z:\Bel\Patryk\Labelling_Marta_Organoid_only\LCNEC11"
ckpt = r"y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\finetunemodel\nxd8_vit_b_lm_finetuned_combined_all_roots_organoidonly.pth"

image_folder_annotator(
    input_folder=rf"{root}\tiff",
    output_folder=rf"{root}\labels",
    pattern="*.tiff",                       # images are .tiff 
    model_type="vit_b",                     # architecture of the exported checkpoint
    checkpoint_path=ckpt,                    # this is our finetuned model
    device=device,
    skip_segmented=False,                   # load existing labels for correction, don't skip them
    is_volumetric=False,
)

# Existing Label Curation
- edit the "labels" layer (fill/bucket set to 0 to delete an object, eraser, paintbrush)
- click "Save & Next" to overwrite that image's .tif and load the next one
- "Save (stay)" saves without advancing; "Prev" goes back without saving

In [ ]:
root = r"Z:\Bel\Patryk\Labelling_Marta_Organoid_only\LCNEC23" # update this to the folder you want to work on 
image_paths = sorted(Path(rf"{root}\tiff").glob("*.tiff"))
label_paths = sorted(Path(rf"{root}\labels").glob("*.tif"))
assert image_paths, "No images found - check the folder / pattern."
assert [p.stem for p in image_paths] == [p.stem for p in label_paths], "image/label names differ"


def load_pair(idx):
    """Load image `idx` and its existing label into the viewer as an editable Labels layer."""
    img = imageio.imread(image_paths[idx])
    lab = imageio.imread(label_paths[idx]).astype(np.int32)
    viewer.layers.clear()
    viewer.add_image(img, name=image_paths[idx].name)
    lbl = viewer.add_labels(lab, name="labels")
    lbl.metadata["idx"] = idx
    lbl.metadata["path"] = label_paths[idx]
    viewer.title = f"[{idx}] {label_paths[idx].name}"
    print(f"editing [{idx}] {label_paths[idx].name} | ids: {np.unique(lab)[1:]}")
    return lbl


# --- GUI panel -----------------------------------------------------------------
_state = {"idx": 0}


def _save():
    lbl = viewer.layers["labels"]
    imageio.imwrite(lbl.metadata["path"], lbl.data.astype(np.uint16))
    print("saved:", lbl.metadata["path"])


def _go(idx):
    idx = max(0, min(idx, len(label_paths) - 1))
    _state["idx"] = idx
    load_pair(idx)
    _status.value = f"{idx + 1} / {len(label_paths)}  -  {label_paths[idx].name}"


def _save_next():
    _save()
    if _state["idx"] < len(label_paths) - 1:
        _go(_state["idx"] + 1)
    else:
        _status.value = f"done - last image ({len(label_paths)}) saved"


_status = Label(value="")
_btn_prev = PushButton(text="< Prev (no save)")
_btn_save = PushButton(text="Save (stay)")
_btn_next = PushButton(text="Save & Next >")
_btn_prev.clicked.connect(lambda: _go(_state["idx"] - 1))
_btn_save.clicked.connect(_save)
_btn_next.clicked.connect(_save_next)

viewer = napari.Viewer()
curation_widget = Container(widgets=[_status, _btn_prev, _btn_save, _btn_next])

# Avoid stacking duplicate panels if this block is re-run.
if "Curate" in viewer.window._dock_widgets:
    viewer.window.remove_dock_widget(viewer.window._dock_widgets["Curate"])
viewer.window.add_dock_widget(curation_widget, area="right", name="Curate")

_go(0)


# Fine Tuning the Model
- fine-tune the Light Microscopy model (vit_b_lm) on your seed labels.

In [2]:
labels_folder_1 = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\ADC15T\labels")
initial_image_folder_1 = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\ADC15T\tiff")

labels_folder_2 = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\LCNEC26\labels")
initial_image_folder_2 = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\LCNEC26\tiff")

labels_folder_3= Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\LCNEC11\labels")
initial_image_folder_3 = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\LCNEC11\tiff")

labels_folder_4 = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\LCNEC23\labels")
initial_image_folder_4 = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\LCNEC23\tiff")

labels_folder_5 = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\LCNEC27\labels")
initial_image_folder_5 = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\LCNEC27\tiff")

labels_folder_6 = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\LNET10d\labels")
initial_image_folder_6 = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\LNET10d\tiff")

labels_folder_7 = Path(r"y:\Patryk_Polinski\NeuroXonnect\Labelling_Marta\extra_images\labels")
initial_image_folder_7 = Path(r"Y:\Patryk_Polinski\NeuroXonnect\Labelling_Marta\extra_images\tiff")

labels_folder_8 = Path(r"y:\Patryk_Polinski\NeuroXonnect\Labelling_Marta\extra_images2\labels")
initial_image_folder_8 = Path(r"Y:\Patryk_Polinski\NeuroXonnect\Labelling_Marta\extra_images2\tiff")

patch_shape = (512, 512)

# this is where the checkpoints + logs are written: <save_root>/checkpoints/<name>/best.pt
save_root = r"Z:\Bel\Patryk\updated_round2"
name = "nxd8_vit_b_lm"

# Match each image to its label by sorted filename order.
image_paths_1 = sorted(initial_image_folder_1.glob("*.tiff"))
label_paths_1 = sorted(labels_folder_1.glob("*.tif"))
image_paths_2 = sorted(initial_image_folder_2.glob("*.tiff"))
label_paths_2= sorted(labels_folder_2.glob("*.tif"))
image_paths_3 = sorted(initial_image_folder_3.glob("*.tiff"))
label_paths_3= sorted(labels_folder_3.glob("*.tif"))
image_paths_4 = sorted(initial_image_folder_4.glob("*.tiff"))
label_paths_4= sorted(labels_folder_4.glob("*.tif"))
image_paths_5 = sorted(initial_image_folder_5.glob("*.tiff"))
label_paths_5= sorted(labels_folder_5.glob("*.tif"))
image_paths_6 = sorted(initial_image_folder_6.glob("*.tiff"))
label_paths_6= sorted(labels_folder_6.glob("*.tif"))
image_paths_7 = sorted(initial_image_folder_7.glob("*.tiff"))
label_paths_7= sorted(labels_folder_7.glob("*.tif"))
image_paths_8 = sorted(initial_image_folder_8.glob("*.tiff"))
label_paths_8= sorted(labels_folder_8.glob("*.tif"))


image_paths = image_paths_1 + image_paths_2 + image_paths_3 + image_paths_4 + image_paths_5 + image_paths_6 + image_paths_7 + image_paths_8
label_paths = label_paths_1 + label_paths_2 + label_paths_3 + label_paths_4 + label_paths_5 + label_paths_6 + label_paths_7 + label_paths_8

print(f"images: {len(image_paths)}  |  labels: {len(label_paths)}")
assert len(image_paths) > 0, "No images found - check the folder / pattern."
assert len(image_paths) == len(label_paths), "image/label counts differ - names must correspond."

# Load each image/label as a separate 2D array.
raw_arrays = [imageio.imread(p).astype(np.int32) for p in image_paths]         # list of 2D arrays (H, W)
label_arrays = [imageio.imread(p).astype(np.int32) for p in label_paths]
print("example image shape:", raw_arrays[0].shape, "| example label shape:", label_arrays[0].shape)
print("raw range:", int(np.min(raw_arrays[0])), "-", int(np.max(raw_arrays[0])))

raw_transform = get_raw_transform("normalize_percentile")

# Pass lists of 2D arrays (raw_key/label_key=None) so each is treated as a separate 2D
# image (image-collection dataset), not slices of a 3D volume. with_channels=False = grayscale.
# min_num_instances=1 keeps single-object images (default is 2).
ds = default_sam_dataset(
    raw_paths=raw_arrays, raw_key=None,
    label_paths=label_arrays, label_key=None,
    patch_shape=patch_shape,
    with_segmentation_decoder=True,
    with_channels=False,
    raw_transform=raw_transform,
    sampler=MinInstanceSampler(min_num_instances=1, min_size=25),
)
n_val = max(1, int(0.1 * len(ds)))
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val])

# num_workers=0 on Windows notebooks avoids multiprocessing/spawn issues.
train_loader = torch_em.get_data_loader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader = torch_em.get_data_loader(val_ds, batch_size=1, shuffle=True, num_workers=0)
print(f"train patches/epoch: {len(train_ds)}  |  val: {len(val_ds)}")


images: 76  |  labels: 76
example image shape: (2160, 2160) | example label shape: (2160, 2160)
raw range: 1478 - 11545
train patches/epoch: 90  |  val: 10


In [3]:
print(len(image_paths_8))

18


In [4]:
# Drop only truly-empty label images (0 objects); single-object images are kept, then rebuild loaders.
def _n_instances(lab):
    return int(np.count_nonzero(np.unique(lab)))   # distinct non-zero label ids

_counts  = [_n_instances(lab) for lab in label_arrays]
_keep    = [i for i, c in enumerate(_counts) if c >= 1]
_dropped = [(image_paths[i].name, _counts[i]) for i in range(len(label_arrays)) if i not in _keep]
if _dropped:
    print(f"dropping {len(_dropped)} image(s) with no labelled objects:")
    for _n, _c in _dropped:
        print(f"   {_n}  ({_c} objects)")
else:
    print("no empty label images found - all pairs kept")

raw_arrays   = [raw_arrays[i]   for i in _keep]
label_arrays = [label_arrays[i] for i in _keep]
assert raw_arrays, "All images were dropped - check that your label files contain instance labels."
print("kept", len(raw_arrays), "image/label pairs")

ds = default_sam_dataset(
    raw_paths=raw_arrays, raw_key=None,
    label_paths=label_arrays, label_key=None,
    patch_shape=patch_shape,
    with_segmentation_decoder=True,
    with_channels=False,
    raw_transform=raw_transform,
    sampler=MinInstanceSampler(min_num_instances=1, min_size=25),
)
n_val = max(1, int(0.1 * len(ds)))
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val])
train_loader = torch_em.get_data_loader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader   = torch_em.get_data_loader(val_ds,   batch_size=1, shuffle=True, num_workers=0)
print(f"train patches/epoch: {len(train_ds)}  |  val: {len(val_ds)}")


dropping 13 image(s) with no labelled objects:
   r03c03f02p01-ch1sk1fk1fl1.tiff  (0 objects)
   r03c03f28p01-ch1sk1fk1fl1.tiff  (0 objects)
   r03c01f12p01-ch1sk1fk1fl1.tiff  (0 objects)
   r01c02f08p01-ch1sk1fk1fl1.tiff  (0 objects)
   r01c02f09p01-ch1sk1fk1fl1.tiff  (0 objects)
   r01c03f02p01-ch1sk1fk1fl1.tiff  (0 objects)
   r01c03f22p01-ch1sk1fk1fl1.tiff  (0 objects)
   r01c04f06p01-ch1sk1fk1fl1.tiff  (0 objects)
   r02c02f48p01-ch1sk1fk1fl1.tiff  (0 objects)
   r03c03f11p01-ch1sk1fk1fl1.tiff  (0 objects)
   r03c04f16p01-ch1sk1fk1fl1.tiff  (0 objects)
   r03c05f06p01-ch1sk1fk1fl1.tiff  (0 objects)
   r04c02f23p01-ch1sk1fk1fl1.tiff  (0 objects)
kept 63 image/label pairs
train patches/epoch: 90  |  val: 10


In [5]:

# Retry-on-empaty: if augmentation empties the instance channel, redraw a new patch.
_DsType = type(ds)
if not getattr(_DsType, "_skip_empty_patched", False):
    _orig_getitem = _DsType.__getitem__

    def _getitem_skip_empty(self, index, _orig=_orig_getitem):
        for _ in range(25):
            x, y = _orig(self, index)
            if np.count_nonzero(np.asarray(y)[0]) > 0:   # channel 0 = instance IDs
                return x, y
            index = np.random.randint(len(self))          # try a different patch
        return x, y                                        # give up (astronomically rare)

    _DsType.__getitem__ = _getitem_skip_empty
    _DsType._skip_empty_patched = True
    print("patched: empty augmented patches will now be skipped")
else:
    print("already patched")

patched: empty augmented patches will now be skipped


In [6]:
# Run fine-tuning.
train_sam(
    name=name,
    model_type="vit_b_lm",          # start from the Light Microscopy model
    train_loader=train_loader,
    val_loader=val_loader,
    n_objects_per_batch=10,        
    with_segmentation_decoder=True,
    save_root=str(save_root),
    n_epochs=100,                   # early stopping halts sooner once val stops improving
)
print("best checkpoint:", save_root / "checkpoints" / name / "best.pt")


Verifying labels in 'val' dataloader:  20%|██        | 10/50 [00:00<00:03, 10.52it/s]


Start fitting for 9000 iterations /  100 epochs
with 90 iterations per epoch
Training with mixed precision


Epoch 14: average [s/it]: 8.694307, current metric: 0.163348, best metric: 0.122455:  15%|█▌        | 1350/9000 [1:46:26<10:03:11,  4.73s/it]

Stopping training because there has been no improvement for 10 epochs
Finished training after 14 epochs / 1350 iterations.
The best epoch is number 3.
Training took 6393.95593047142 seconds (= 01:106:34 hours)


TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [ ]:
# Step 3 — export the fine-tuned model so it works in the napari GUI / CLI / library,
# then re-open the annotator using it. In the GUI, put this path in
# Embedding Settings -> "custom weights path" (keep Model = Light Microscopy, size = base).

best_ckpt = r"Z:\Bel\Patryk\updated_round2\checkpoints\nxd8_vit_b_lm\best.pt"
# finetuned_path = save_root / f"{name}_finetuned_combined_two_roots.pth"
finetuned_path = r"Z:\Bel\Patryk\updated_round2\finetuned_combined_round2updates_clean.pth"

export_custom_sam_model(
    checkpoint_path=str(best_ckpt),
    model_type="vit_b",              # architecture size only (base); the LM weights are inside best.pt
    save_path=str(finetuned_path),
    with_segmentation_decoder=True,  # keep the decoder so AIS/APG work
)
print("finetuned model exported to:", finetuned_path)


In [ ]:
unseen_folders = [
    Path(r"y:\Patryk_Polinski\NeuroXonnect\NXD8\24wp\PP_NXD8_129_DIV18_18102025\Images"),
    Path(r"y:\Patryk_Polinski\NeuroXonnect\NXD8\24wp\PP_NXD8_129_DIV16_16102025\Images"),
    Path(r"y:\Patryk_Polinski\NeuroXonnect\NXD8\24wp\PP_NXD8_129_DIV25_25102025\Images"),
    Path(r"y:\Patryk_Polinski\NeuroXonnect\NXD8\24wp\PP_NXD8_129_DIV26_26102025\Images"),
    Path(r"y:\Patryk_Polinski\NeuroXonnect\NXD11\CoCult\PP_NXD11_129_DIV27_17112025\Images"),
         ]

titles = ["NXD8_129_DIV18_18102025", "NXD8_129_DIV16_16102025", "NXD8_129_DIV25_25102025", "NXD8_129_DIV26_26102025", "NXD11_129_DIV27_17112025"]

for j, unseen_folder in enumerate(unseen_folders):
    unseen_paths = sorted(unseen_folder.rglob("*.tiff"))
    print(f"found {len(unseen_paths)} images in {unseen_folder.name}")

    if not unseen_paths:
        print("No images found - nothing to segment.")
    else:
        # --- pick a random sample ---
        n_show = min(20, len(unseen_paths))
        sample_paths = random.sample(unseen_paths, n_show)

    # --- load the fine-tuned model + instance-seg decoder once (cached across re-runs) ---
    if "_ft_segmenter" not in globals():
        _ft_predictor, _ft_segmenter = get_predictor_and_segmenter(
            model_type="vit_b",                # architecture of the exported checkpoint
            checkpoint=str(r"z:\Bel\Patryk\updated_round2\finetuned_combined_round2updates.pth"),    # your fine-tuned weights (with decoder)
            device=device,
            segmentation_mode="ais",           # automatic instance segmentation via the decoder
        )

    def _to_gray(a):
        a = np.squeeze(a)
        if a.ndim == 3:                        # (H,W,C) or (C,H,W) -> collapse to single 2D channel
            a = a.mean(axis=int(np.argmin(a.shape)))
        return a
    print("HERE")
    # --- segment + plot (same percentile normalization the model was trained with) ---
    fig, axes = plt.subplots(n_show, 2, figsize=(11, 5.2 * n_show), squeeze=False)
    for i, p in enumerate(sample_paths):
        img_norm = raw_transform(_to_gray(imageio.imread(p)).astype(np.float32))   # -> [0, 255]
        seg = automatic_instance_segmentation(
            predictor=_ft_predictor, segmenter=_ft_segmenter,
            input_path=img_norm, ndim=2, verbose=False,
        )
        overlay = label2rgb(seg, image=img_norm / 255.0, bg_label=0, alpha=0.45)
        axes[i, 0].imshow(img_norm, cmap="gray")
        axes[i, 0].set_title(p.name, fontsize=9); axes[i, 0].axis("off")
        axes[i, 1].imshow(overlay)
        axes[i, 1].set_title(f"{int(seg.max())} objects", fontsize=9); axes[i, 1].axis("off")
    fig.suptitle("Fine-tuned micro-sam (AIS) on unseen images", y=1.0)
    plt.tight_layout()
    plt.savefig(rf"Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\unseen_predictions\{titles[j]}_secondupdates_outputs_fullmodel.png")

In [13]:
from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as imageio
import torch

from skimage.color import label2rgb

from micro_sam.automatic_segmentation import (
    get_predictor_and_segmenter,
    automatic_instance_segmentation,
)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint = Path(
    r"z:\Bel\Patryk\updated_round2\finetuned_combined_round2updates.pth"
)

output_folder = Path(
    r"Y:\Patryk_Polinski\Bel_Marta_Classification"
    r"\Labelling_Marta_Organoid_only\unseen_predictions"
)

output_folder.mkdir(parents=True, exist_ok=True)


unseen_folders = [
    Path(r"y:\Patryk_Polinski\NeuroXonnect\NXD8\24wp\PP_NXD8_129_DIV18_18102025\Images"),
    Path(r"y:\Patryk_Polinski\NeuroXonnect\NXD8\24wp\PP_NXD8_129_DIV16_16102025\Images"),
    Path(r"y:\Patryk_Polinski\NeuroXonnect\NXD8\24wp\PP_NXD8_129_DIV25_25102025\Images"),
    Path(r"y:\Patryk_Polinski\NeuroXonnect\NXD8\24wp\PP_NXD8_129_DIV26_26102025\Images"),
    Path(r"y:\Patryk_Polinski\NeuroXonnect\NXD11\CoCult\PP_NXD11_129_DIV27_17112025\Images"),
]

titles = [
    "NXD8_129_DIV18_18102025",
    "NXD8_129_DIV16_16102025",
    "NXD8_129_DIV25_25102025",
    "NXD8_129_DIV26_26102025",
    "NXD11_129_DIV27_17112025",
]


# Thresholds to compare
foreground_thresholds = [
    0.5,
    0.6,
    0.7,
    0.8,
]


# Keep min_size at zero initially.
# This isolates the effect of foreground_threshold.
min_size = 0


# Number of images to test from each folder
N_SHOW = 20


# Make random selection reproducible
random.seed(42)


# ------------------------------------------------------------------
# HELPER
# ------------------------------------------------------------------

def _to_gray(a):
    a = np.squeeze(a)

    if a.ndim == 3:
        # (H,W,C) or (C,H,W) -> collapse to single 2D channel
        a = a.mean(axis=int(np.argmin(a.shape)))

    return a


# ------------------------------------------------------------------
# LOAD MODEL ONCE
# ------------------------------------------------------------------

print(f"Using device: {device}")
print("Loading fine-tuned micro-sam model...")


_ft_predictor, _ft_segmenter = get_predictor_and_segmenter(
    model_type="vit_b",
    checkpoint=str(checkpoint),
    device=device,
    segmentation_mode="ais",
)


print("Model loaded.")


# ------------------------------------------------------------------
# TEST EACH DATASET
# ------------------------------------------------------------------

for j, unseen_folder in enumerate(unseen_folders):

    unseen_paths = sorted(unseen_folder.rglob("*.tiff"))

    print()
    print("=" * 80)
    print(f"{titles[j]}")
    print(f"Found {len(unseen_paths)} images")
    print("=" * 80)

    if not unseen_paths:
        print("No images found - skipping.")
        continue


    # --------------------------------------------------------------
    # Pick SAME random images for all thresholds
    # --------------------------------------------------------------

    n_show = min(N_SHOW, len(unseen_paths))

    sample_paths = random.sample(
        unseen_paths,
        n_show,
    )


    # --------------------------------------------------------------
    # Plot:
    #
    # column 0 = original
    # column 1 = threshold 0.5
    # column 2 = threshold 0.6
    # column 3 = threshold 0.7
    # column 4 = threshold 0.8
    # --------------------------------------------------------------

    n_cols = 1 + len(foreground_thresholds)

    fig, axes = plt.subplots(
        n_show,
        n_cols,
        figsize=(4.5 * n_cols, 4.5 * n_show),
        squeeze=False,
    )


    # --------------------------------------------------------------
    # Process images
    # --------------------------------------------------------------

    for i, p in enumerate(sample_paths):

        print(
            f"[{i + 1:02d}/{n_show:02d}] "
            f"{p.name}"
        )


        # ----------------------------------------------------------
        # Load + same normalization as training
        # ----------------------------------------------------------

        raw = imageio.imread(p)

        img_gray = _to_gray(raw).astype(np.float32)

        img_norm = raw_transform(img_gray)


        # ----------------------------------------------------------
        # Original image
        # ----------------------------------------------------------

        axes[i, 0].imshow(
            img_norm,
            cmap="gray",
        )

        axes[i, 0].set_title(
            f"{p.name}\nOriginal",
            fontsize=8,
        )

        axes[i, 0].axis("off")


        # ----------------------------------------------------------
        # Run each foreground threshold
        # ----------------------------------------------------------

        for k, threshold in enumerate(foreground_thresholds):

            print(
                f"    foreground_threshold = {threshold}"
            )


            seg = automatic_instance_segmentation(
                predictor=_ft_predictor,
                segmenter=_ft_segmenter,
                input_path=img_norm,
                ndim=2,
                verbose=False,

                # ----------------------------------------------
                # AIS PARAMETERS
                # ----------------------------------------------

                foreground_threshold=threshold,

                # Keep these fixed at their standard values
                center_distance_threshold=0.5,
                boundary_distance_threshold=0.5,
                foreground_smoothing=1.0,
                distance_smoothing=1.6,

                # Don't filter based on size yet
                min_size=min_size,
            )


            # Number of instances
            object_ids = np.unique(seg)
            object_ids = object_ids[object_ids != 0]

            n_objects = len(object_ids)


            # --------------------------------------------------
            # Overlay
            # --------------------------------------------------

            overlay = label2rgb(
                seg,
                image=img_norm / 255.0,
                bg_label=0,
                alpha=0.45,
            )


            axes[i, k + 1].imshow(overlay)

            axes[i, k + 1].set_title(
                f"threshold = {threshold}\n"
                f"{n_objects} objects",
                fontsize=9,
            )

            axes[i, k + 1].axis("off")


    # --------------------------------------------------------------
    # Save comparison
    # --------------------------------------------------------------

    fig.suptitle(
        f"{titles[j]}\n"
        f"Fine-tuned micro-sam AIS: foreground threshold comparison",
        fontsize=16,
        y=1.0,
    )

    plt.tight_layout()


    save_path = (
        output_folder /
        f"{titles[j]}_foreground_threshold_comparison.png"
    )


    plt.savefig(
        save_path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close(fig)


    print()
    print(f"Saved:")
    print(save_path)


print()
print("DONE")

Using device: cuda
Loading fine-tuned micro-sam model...
Model loaded.

NXD8_129_DIV18_18102025
Found 1200 images
[01/20] r01c05f33p01-ch1sk1fk1fl1.tiff
    foreground_threshold = 0.5


    foreground_threshold = 0.6


    foreground_threshold = 0.7


    foreground_threshold = 0.8


[02/20] r01c02f03p01-ch1sk1fk1fl1.tiff
    foreground_threshold = 0.5


...

[20/20] r02c06f13p01-ch1sk1fk1fl1.tiff
    foreground_threshold = 0.5


    foreground_threshold = 0.6


    foreground_threshold = 0.7


    foreground_threshold = 0.8



Saved:
Y:\Patryk_Polinski\Bel_Marta_Classification\Labelling_Marta_Organoid_only\unseen_predictions\NXD11_129_DIV27_17112025_foreground_threshold_comparison.png

DONE
